# 01 — Data Collection and EDA

Run from top to bottom. Inspect the actual schema before selecting target/predictor columns. The SpaceX API is live and may differ from the course dataset.

In [ ]:
from pathlib import Path
import requests, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
Path('../data').mkdir(exist_ok=True)
url = 'https://api.spacexdata.com/v4/launches/past'
r = requests.get(url, timeout=45)
r.raise_for_status()
raw = r.json()
df = pd.json_normalize(raw)
print('Shape:', df.shape)
print(df.columns.tolist()[:60])
df.head()

## Data quality checks

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
missing = df.isna().sum().sort_values(ascending=False)
display(missing.head(25).to_frame('missing_count'))
print(df.dtypes)

## Example temporal EDA

This chart describes records in the downloaded API response; it does not define landing success.

In [ ]:
date_col = 'date_utc' if 'date_utc' in df.columns else None
if date_col:
    dates = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    year_counts = dates.dt.year.value_counts().sort_index()
    ax = year_counts.plot(kind='bar', figsize=(10,4), title='SpaceX API launch records by year')
    ax.set_xlabel('Year'); ax.set_ylabel('Record count'); plt.tight_layout(); plt.show()
else:
    print('No date_utc column found; inspect the schema and choose the correct date field.')

## Save raw snapshot and normalized data

In [ ]:
df.to_csv('../data/launches_past_normalized.csv', index=False)
print('Saved normalized data. Record retrieval date and source URL in your report.')